# মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক সহ হিউম্যান-ইন-দ্য-লুপ ওয়ার্কফ্লো

## 🎯 শেখার উদ্দেশ্য

এই নোটবুকে, আপনি শিখবেন কিভাবে মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্কের `RequestInfoExecutor` ব্যবহার করে **হিউম্যান-ইন-দ্য-লুপ** ওয়ার্কফ্লো বাস্তবায়ন করতে হয়। এই শক্তিশালী প্যাটার্নটি আপনাকে মানব ইনপুট সংগ্রহের জন্য এআই ওয়ার্কফ্লো স্থগিত করতে দেয়, যার ফলে আপনার এজেন্টগুলি ইন্টারেক্টিভ হয় এবং মানুষেরা গুরুত্বপূর্ণ সিদ্ধান্তে নিয়ন্ত্রণ পায়।

## 🔄 হিউম্যান-ইন-দ্য-লুপ কী?

**হিউম্যান-ইন-দ্য-লুপ (HITL)** একটি ডিজাইন প্যাটার্ন যেখানে এআই এজেন্টগুলি চলাকালীন মানব ইনপুটের জন্য স্থগিত করে। এটি প্রয়োজনীয়:

- ✅ **গুরুত্বপূর্ণ সিদ্ধান্ত** - গুরুত্বপূর্ণ পদক্ষেপের আগে মানুষের অনুমোদন নিন
- ✅ **অস্পষ্ট পরিস্থিতি** - যখন এআই অনিশ্চিত তখন মানুষ স্পষ্টকরণ দেয়
- ✅ **ব্যবহারকারীর পছন্দসমূহ** - একাধিক বিকল্প থেকে ব্যবহারকারীদের চয়ন করতে বলুন
- ✅ **কমপ্লায়েন্স এবং নিরাপত্তা** - নিয়ন্ত্রিত কার্যক্রমে মানুষের তত্ত্বাবধান নিশ্চিত করুন
- ✅ **ইন্টারেক্টিভ অভিজ্ঞতা** - ব্যবহারকারীর ইনপুটে প্রতিক্রিয়া জানানো কথোপকথনমূলক এজেন্ট তৈরি করুন

## 🏗️ মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্কে এটি কীভাবে কাজ করে

ফ্রেমওয়ার্কটি HITL এর জন্য তিনটি প্রধান উপাদান প্রদান করে:

1. **`RequestInfoExecutor`** - একটি বিশেষ নির্বাহকারী যা ওয়ার্কফ্লো স্থগিত করে এবং একটি `RequestInfoEvent` প্রেরণ করে
2. **`RequestInfoMessage`** - মানুষের কাছে প্রেরিত টাইপ করা অনুরোধের মূল শ্রেণি
3. **`RequestResponse`** - `request_id` ব্যবহার করে মানুষের প্রতিক্রিয়াকে মূল অনুরোধের সাথে সামঞ্জস্য করে

**ওয়ার্কফ্লো প্যাটার্ন:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 আমাদের উদাহরণ: ব্যবহারকারীর নিশ্চিতকরণের সাথে হোটেল বুকিং

আমরা শর্তাধীন ওয়ার্কফ্লোর উপর ভিত্তি করে বিকল্প গন্তব্যের পরামর্শ দেওয়ার **পূর্বে** মানুষের নিশ্চিতকরণ যোগ করব:

1. ব্যবহারকারী একটি গন্তব্য অনুরোধ করে (যেমন, "প্যারিস")
2. `availability_agent` চেক করে রুম উপলব্ধ কিনা
3. **যদি কোনো রুম না থাকে** → `confirmation_agent` জিজ্ঞাসা করে "আপনি বিকল্প দেখতে চান কি?"
4. ওয়ার্কফ্লো **স্থগিত হয়** `RequestInfoExecutor` ব্যবহার করে
5. **মানুষ "হ্যাঁ" বা "না" বলে প্রতিক্রিয়া দেয়** কনসোল ইনপুটের মাধ্যমে
6. `decision_manager` প্রতিক্রিয়ার উপর ভিত্তি করে পথ নির্ধারণ করে:
   - **হ্যাঁ** → বিকল্প গন্তব্য দেখান
   - **না** → বুকিং অনুরোধ বাতিল করুন
7. চূড়ান্ত ফলাফল প্রদর্শন করুন

এটি দেখায় কিভাবে ব্যবহারকারীদের এজেন্টের পরামর্শের উপর নিয়ন্ত্রণ দেওয়া যায়!

---

চলুন শুরু করা যাক! 🚀


## ধাপ ১: প্রয়োজনীয় লাইব্রেরি আমদানী

আমরা স্ট্যান্ডার্ড এজেন্ট ফ্রেমওয়ার্ক উপাদানগুলো আমদানি করি পাশাপাশি **মানুষ-সম্পৃক্ত নির্দিষ্ট ক্লাসগুলো**:
- `RequestInfoExecutor` - এমন এক্সিকিউটর যা মানুষের ইনপুটের জন্য ওয়ার্কফ্লো থামায়
- `RequestInfoEvent` - যখন মানুষের ইনপুট চাওয়া হয় তখন ইভেন্ট সৃষ্টি হয়
- `RequestInfoMessage` - টাইপ করা অনুরোধ পে-লোডের বেস ক্লাস
- `RequestResponse` - মানুষের প্রতিক্রিয়া অনুরোধের সাথে সম্পৃক্ত করে
- `WorkflowOutputEvent` - ওয়ার্কফ্লো আউটপুট সনাক্ত করার ইভেন্ট


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## পদক্ষেপ ২: কাঠামোগত আউটপুটের জন্য পাইডান্টিক মডেল সংজ্ঞায়িত করা

এই মডেলগুলো সংজ্ঞায়িত করে সেই **স্কিমা** যা এজেন্টরা ফিরিয়ে দেবে। আমরা শর্তসাপেক্ষ ওয়ার্কফ্লো থেকে সব মডেল রাখি এবং যোগ করি:

**হিউম্যান-ইন-দ্য-লুপের জন্য নতুন:**
- `HumanFeedbackRequest` - `RequestInfoMessage` এর সাবক্লাস যা মানুষের কাছে পাঠানো অনুরোধের পে-লোড নির্দিষ্ট করে
  - এতে থাকে `prompt` (প্রশ্ন যা জিজ্ঞাসা করা হবে) এবং `destination` (অপ্রাপ্য শহর সম্পর্কিত প্রেক্ষাপট)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## ধাপ ৩: হোটেল বুকিং টুল তৈরি করুন

শর্তাধীন ওয়ার্কফ্লো থেকে একই টুল - গন্তব্যে ঘর পাওয়া যায় কি না তা পরীক্ষা করে।


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## ধাপ ৪: রাউটিংয়ের জন্য শর্ত ফাংশন নির্ধারণ

আমাদের মানব-ইন-দ্য-লুপ ওয়ার্কফ্লোর জন্য **চারটি শর্ত ফাংশনের** প্রয়োজন:

**শর্তযুক্ত ওয়ার্কফ্লো থেকে:**
১. `has_availability_condition` - যখন হোটেল উপলব্ধ হয় তখন রাউট করে
২. `no_availability_condition` - যখন হোটেল উপলব্ধ নয় তখন রাউট করে

**মানব-ইন-দ্য-লুপের জন্য নতুন:**
৩. `user_wants_alternatives_condition` - যখন ব্যবহারকারী বিকল্প সমর্থন করেন তখন রাউট করে
৪. `user_declines_alternatives_condition` - যখন ব্যবহারকারী বিকল্প অস্বীকার করেন তখন রাউট করে


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## ধাপ ৫: ডিসিশন ম্যানেজার এক্সিকিউটর তৈরি করুন

এটি হলো **হিউম্যান-ইন-দ্য-লুপ প্যাটার্নের মূল অংশ**! `DecisionManager` একটি কাস্টম `Executor` যা:

১. `RequestResponse` অবজেক্টের মাধ্যমে **মানব প্রতিক্রিয়া গ্রহণ করে**
২. **ব্যবহারকারীর সিদ্ধান্ত প্রক্রিয়া করে** (হ্যাঁ/না)
৩. **সঠিক এজেন্টকে বার্তা পাঠিয়ে ওয়ার্কফ্লো রাউট করে**

প্রধান বৈশিষ্ট্যসমূহ:
- মেথডগুলোকে ওয়ার্কফ্লো ধাপ হিসেবে প্রকাশ করতে `@handler` ডেকোরেটর ব্যবহার করে
- `RequestResponse[HumanFeedbackRequest, str]` গ্রহণ করে, যাতে থাকে মূল অনুরোধ এবং ব্যবহারকারীর উত্তর
- সহজ "হ্যাঁ" বা "না" বার্তা প্রদান করে যা আমাদের শর্ত ফাংশনগুলোকে ট্রিগার করে


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## ধাপ ৬: কাস্টম ডিসপ্লে এক্সিকিউটর তৈরি করুন

শর্তাধীন ওয়ার্কফ্লো থেকে একই ডিসপ্লে এক্সিকিউটর - ওয়ার্কফ্লো আউটপুট হিসাবে চূড়ান্ত ফলাফল দেবে।


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## ধাপ ৭: পরিবেশ ভেরিয়েবলগুলি লোড করুন

LLM ক্লায়েন্ট (Microsoft Foundry / Azure OpenAI) কনফিগার করুন।


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## ধাপ ৮: AI এজেন্ট এবং এক্সিকিউটর তৈরি করুন

আমরা **ছয়টি ওয়ার্কফ্লো কম্পোনেন্ট** তৈরি করি:

**এজেন্টস (AgentExecutor এর মধ্যে মোড়ানো):**
১. **availability_agent** - সরঞ্জামটি ব্যবহার করে হোটেলের প্রাপ্যতা পরীক্ষা করে
২. **confirmation_agent** - 🆕 মানব নিশ্চিতকরণ অনুরোধ প্রস্তুত করে
৩. **alternative_agent** - বিকল্প শহর পরামর্শ দেয় (যখন ব্যবহারকারী হ্যাঁ বলে)
৪. **booking_agent** - বুকিং করতে উৎসাহিত করে (যখন কক্ষ পাওয়া যায়)
৫. **cancellation_agent** - 🆕 বাতিলকরণের বার্তা পরিচালনা করে (যখন ব্যবহারকারী না বলে)

**বিশেষ এক্সিকিউটরস:**
৬. **request_info_executor** - 🆕 `RequestInfoExecutor` যা মানব ইনপুটের জন্য ওয়ার্কফ্লো থামায়
৭. **decision_manager** - 🆕 কাস্টম এক্সিকিউটর যা মানব প্রতিক্রিয়ার উপর ভিত্তি করে রুট করে (আগেই সংজ্ঞায়িত হয়েছে)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## ধাপ ৯: মানব-সম্পৃক্ত লুপ সহ ওয়ার্কফ্লো তৈরি করা

এখন আমরা **শর্তাধীন রাউটিং** সহ মানব-সম্পৃক্ত লুপ পাথসহ ওয়ার্কফ্লো গ্রাফ নির্মাণ করব:

**ওয়ার্কফ্লো কাঠামো:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**মূল সংযোগগুলি:**
- `availability_agent → confirmation_agent` (যখন কোনো রুম নেই)
- `confirmation_agent → prepare_human_request` (রূপান্তর প্রকার)
- `prepare_human_request → request_info_executor` (মানবের জন্য বিরতি)
- `request_info_executor → decision_manager` (সবসময় - RequestResponse প্রদান করে)
- `decision_manager → alternative_agent` (যখন ব্যবহারকারী "হ্যাঁ" বলেন)
- `decision_manager → cancellation_agent` (যখন ব্যবহারকারী "না" বলেন)
- `availability_agent → booking_agent` (যখন রুম উপলব্ধ)
- সকল পথ শেষ হয় `display_result` এ


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## ধাপ ১০: টেস্ট কেস ১ চালান - শহর যা অ্যাভেলেবিলিটি ছাড়াই (প্যারিস সাথে মানব নিশ্চিতকরণ)

এই টেস্ট **সম্পূর্ণ মানব-ইন-দ্য-লুপ চক্র** প্রদর্শন করে:

১. প্যারিসে হোটেলের অনুরোধ
২. availability_agent চেক করে → কোনো রুম নেই
৩. confirmation_agent মানব-মুখী প্রশ্ন তৈরি করে
৪. request_info_executor **ওয়ার্কফ্লো স্থগিত করে** এবং `RequestInfoEvent` ইমিট করে
৫. **অ্যাপ্লিকেশন ইভেন্ট সনাক্ত করে এবং কনসোলে ব্যবহারকারীকে প্রম্পট করে**
৬. ব্যবহারকারী "yes" বা "no" টাইপ করে
৭. অ্যাপ্লিকেশন `send_responses_streaming()` এর মাধ্যমে প্রতিক্রিয়া পাঠায়
৮. decision_manager প্রতিক্রিয়ার উপর ভিত্তি করে রুটিং করে
৯. চূড়ান্ত ফলাফল প্রদর্শিত হয়

**মূল প্যাটার্ন:**
- প্রথম ইটারেশনের জন্য `workflow.run_stream()` ব্যবহার করুন
- পরবর্তী ইটারেশনের জন্য `workflow.send_responses_streaming(pending_responses)` ব্যবহার করুন
- মানব ইনপুট দরকার হলে তা সনাক্ত করতে `RequestInfoEvent` এর জন্য শুনুন
- চূড়ান্ত ফলাফল ক্যাপচার করতে `WorkflowOutputEvent` এর জন্য শুনুন


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## ধাপ ১১: টেস্ট কেস ২ চালান - সিটি সহ উপলব্ধতা (স্টকহোম - কোনো মানুষের ইনপুটের দরকার নেই)

এই পরীক্ষা দেখায় **সরাসরি পথ** যখন কক্ষগুলি উপলব্ধ থাকে:

১. স্টকহোমে হোটেল অনুরোধ করুন
২. availability_agent পরীক্ষা করে → কক্ষগুলি উপলব্ধ ✅
৩. booking_agent বুকিং পরামর্শ দেয়
৪. display_result নিশ্চিতকরণ প্রদর্শন করে
৫. **কোনো মানুষের ইনপুটের দরকার নেই!**

যখন কক্ষগুলি উপলব্ধ থাকে তখন ওয়ার্কফ্লোটি পুরোপুরি মানুষের অংশগ্রহণ ছাড়াই চলে।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## মূল বিষয়সমূহ এবং মানব-ইন-দ্য-লুপ সেরা অনুশীলন

### ✅ আপনি যা শিখেছেন:

#### ১. **RequestInfoExecutor প্যাটার্ন**
Microsoft Agent Framework-এ মানব-ইন-দ্য-লুপ প্যাটার্ন তিনটি মূল উপাদান ব্যবহার করে:
- `RequestInfoExecutor` - ওয়ার্কফ্লো থামায় এবং ইভেন্ট সরবরাহ করে
- `RequestInfoMessage` - টাইপ করা রিকোয়েস্ট পেইলোডের জন্য বেস ক্লাস (এটি সাবক্লাস করুন!)
- `RequestResponse` - মানবিক প্রতিক্রিয়াগুলোকে মূল রিকোয়েস্টের সাথে মিলায়

**গুরুত্বপূর্ণ বোঝাপড়া:**
- `RequestInfoExecutor` নিজেই ইনপুট সংগ্রহ করে না - এটি শুধু ওয়ার্কফ্লো থামায়
- আপনার অ্যাপ্লিকেশন কোডকে `RequestInfoEvent` শুনতে হবে এবং ইনপুট সংগ্রহ করতে হবে
- আপনাকে `send_responses_streaming()` কল করতে হবে একটি ডিকশনারি সহ যেখানে `request_id` ব্যবহারকারীর উত্তরের সাথে ম্যাপ করা হয়

#### ২. **স্ট্রিমিং এক্সিকিউশন প্যাটার্ন**
```python
# প্রথম পুনরাবৃত্তি
stream = workflow.run_stream(initial_request)

# পরবর্তী পুনরাবৃত্তি (মানব ইনপুটের পরে)
stream = workflow.send_responses_streaming(pending_responses)

# সর্বদা ইভেন্টগুলি প্রক্রিয়া করুন
events = [event async for event in stream]
```

#### ৩. **ইভেন্ট-চালিত আর্কিটেকচার**
নির্দিষ্ট ইভেন্টের জন্য শুনুন ওয়ার্কফ্লো নিয়ন্ত্রণ করতে:
- `RequestInfoEvent` - মানবিক ইনপুট দরকার (ওয়ার্কফ্লো থামানো হয়েছে)
- `WorkflowOutputEvent` - চূড়ান্ত ফলাফল উপলব্ধ (ওয়ার্কফ্লো সম্পন্ন)
- `WorkflowStatusEvent` - স্টেট পরিবর্তন (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, ইত্যাদি)

#### ৪. **@handler সহ কাস্টম এক্সিকিউটর**
`DecisionManager` দেখায় কিভাবে এক্সিকিউটর তৈরি করবেন যেগুলো:
- `@handler` ডেকোরেটর ব্যবহার করে মেথডগুলোকে ওয়ার্কফ্লো স্টেপ হিসেবে প্রকাশ করে
- টাইপ করা মেসেজ গ্রহণ করে (যেমন, `RequestResponse[HumanFeedbackRequest, str]`)
- মেসেজ পাঠিয়ে অন্য এক্সিকিউটরদের মাধ্যমে ওয়ার্কফ্লো রাউট করে
- `WorkflowContext` এর মাধ্যমে কনটেক্সট অ্যাক্সেস করে

#### ৫. **মানবিক সিদ্ধান্তের সাথে শর্তাধীন রাউটিং**
আপনি এমন শর্ত ফাংশন তৈরি করতে পারেন যা মানবিক প্রতিক্রিয়া মূল্যায়ন করে:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 বাস্তব-বিশ্বের অ্যাপ্লিকেশন:

১. **অনুমোদন ওয়ার্কফ্লো**
   - ব্যয়ের রিপোর্ট প্রক্রিয়াকরণের আগে ম্যানেজারের অনুমোদন নিন
   - স্বয়ংক্রিয় ইমেইল পাঠানোর আগে মানব চেকের প্রয়োজন
   - উচ্চ মূল্যের লেনদেনের আগে নিশ্চিতকরণ নিন

২. **কন্টেন্ট মনিটরিং**
   - সন্দেহজনক কন্টেন্ট চিহ্নিত করুন মানব যাচাইয়ের জন্য
   - এজ কেসে চূড়ান্ত সিদ্ধান্ত নেওয়ার জন্য মডারেটরদের জিজ্ঞাসা করুন
   - যখন AI এর আত্মবিশ্বাস কম হয় তখন সেটি মানুষের কাছে প্রেরণ করুন

৩. **কাস্টমার সার্ভিস**
   - AI স্বয়ংক্রিয়ভাবে রুটিন প্রশ্ন হ্যান্ডেল করুক
   - জটিল সমস্যাগুলো মানুষের কাছে তোলা হোক
   - গ্রাহকের কাছ থেকে জিজ্ঞাসা করুন তারা কি মানুষের সাথে কথা বলতে চায়

৪. **ডাটা প্রক্রিয়াকরণ**
   - অস্পষ্ট ডাটা এন্ট্রি সমাধানের জন্য মানুষের সাহায্য নিন
   - অস্পষ্ট নথির AI ব্যাখ্যা নিশ্চিত করুন
   - ব্যবহারকারীদের একাধিক বৈধ ব্যাখ্যার মধ্যে নির্বাচন করতে দিন

৫. **নিরাপত্তা-সংবেদনশীল সিস্টেম**
   - অপরিবর্তনীয় কাজের আগে মানব নিশ্চিতকরণ আবশ্যক
   - সংবেদনশীল ডেটা অ্যাক্সেস করার আগে অনুমোদন নিন
   - নিয়ন্ত্রিত শিল্পক্ষেত্রে সিদ্ধান্ত নিশ্চিত করুন (স্বাস্থ্যসেবা, অর্থনীতি)

৬. **ইন্টারেক্টিভ এজেন্টস**
   - কথোপকথন চালানো বট তৈরি করুন যা অনুস্মারক প্রশ্ন করে
   - উইজার্ড তৈরি করুন যা জটিল প্রক্রিয়ায় ব্যবহারকারীদের গাইড করে
   - এমন এজেন্ট ডিজাইন করুন যারা ধাপে ধাপে মানুষের সাথে সহযোগিতা করে

### 🔄 তুলনা: মানব-ইন-দ্য-লুপ সহ বনাম ছাড়া

| বৈশিষ্ট্য | শর্তাধীন ওয়ার্কফ্লো | মানব-ইন-দ্য-লুপ ওয়ার্কফ্লো |
|---------|---------------------|---------------------------|
| **এক্সিকিউশন** | একক `workflow.run()` | `run_stream()` + `send_responses_streaming()` সহ লুপ |
| **ব্যবহারকারীর ইনপুট** | নেই (পুরোপুরি স্বয়ংক্রিয়) | ইন্টারেক্টিভ প্রম্পট `input()` বা UI এর মাধ্যমে |
| **উপাদান** | এজেন্ট + এক্সিকিউটর | + RequestInfoExecutor + DecisionManager |
| **ইভেন্টসমূহ** | শুধুমাত্র AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent ইত্যাদি |
| **অস্থায়ী থামানো** | থামানো হয় না | ওয়ার্কফ্লো থামে RequestInfoExecutor এ |
| **মানব নিয়ন্ত্রণ** | মানব নিয়ন্ত্রণ নেই | মানুষ মূল সিদ্ধান্ত নেয় |
| **ব্যবহার ক্ষেত্র** | স্বয়ংক্রিয়তা | সহযোগিতা ও তদারকি |

### 🚀 উন্নত প্যাটার্ন:

#### একাধিক মানব সিদ্ধান্তের পয়েন্ট
একই ওয়ার্কফ্লোতে একাধিক `RequestInfoExecutor` নোড থাকতে পারে:
```python
.add_edge(agent1, request_info_1)  # প্রথম মানব সিদ্ধান্ত
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # দ্বিতীয় মানব সিদ্ধান্ত
.add_edge(decision_manager_2, final_agent)
```

#### টাইমআউট হ্যান্ডলিং
মানব প্রতিক্রিয়ার জন্য টাইমআউট প্রয়োগ করুন:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # নিরাপদ বিকল্পে ডিফল্ট করুন
```

#### সমৃদ্ধ UI ইন্টিগ্রেশন
`input()` এর পরিবর্তে ওয়েব UI, Slack, Teams ইত্যাদির সাথে ইন্টিগ্রেট করুন:
```python
if isinstance(event, RequestInfoEvent):
    # ব্যবহারকারীর পছন্দসই চ্যানেলে বিজ্ঞপ্তি পাঠান
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### শর্তাধীন মানব-ইন-দ্য-লুপ
শুধুমাত্র নির্দিষ্ট পরিস্থিতিতে মানব ইনপুট চাইুন:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # শুধুমাত্র যখন আত্মবিশ্বাস কম বা মান বেশি তখনই মানবকে রুট করুন
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ সেরা অনুশীলন:

১. **সবসময় RequestInfoMessage সাবক্লাস করুন**
   - টাইপ সেফটি এবং যাচাই নিশ্চিত করে
   - UI রেন্ডারিংয়ের জন্য সমৃদ্ধ প্রসঙ্গ সরবরাহ করে
   - প্রতিটি রিকোয়েস্ট টাইপের উদ্দেশ্য স্পষ্ট করে

২. **বর্ণনামূলক প্রম্পট ব্যবহার করুন**
   - আপনি যা চাইছেন তার প্রসঙ্গ অন্তর্ভুক্ত করুন
   - প্রতিটি পছন্দের পরিণতি ব্যাখ্যা করুন
   - প্রশ্নগুলো সহজ এবং স্পষ্ট রাখুন

৩. **অপ্রত্যাশিত ইনপুট পরিচালনা করুন**
   - ব্যবহারকারীর প্রতিক্রিয়া যাচাই করুন
   - অবৈধ ইনপুটের জন্য ডিফল্ট প্রদান করুন
   - স্পষ্ট ত্রুটি বার্তা দিন

৪. **রিকোয়েস্ট আইডি ট্র্যাক করুন**
   - request_id এবং প্রতিক্রিয়ার মধ্যে সম্পর্ক ব্যবহার করুন
   - হাতে স্টেট ম্যানেজ করার চেষ্টা করবেন না

৫. **না-ব্লকিং ডিজাইন করুন**
   - ইনপুটের জন্য অপেক্ষায় থ্রেড ব্লক করবেন না
   - সার্বত্রিক অ্যাসিঙ্ক প্যাটার্ন ব্যবহার করুন
   - সমান্তরাল ওয়ার্কফ্লো ইন্সট্যান্স সমর্থন করুন

### 📚 সম্পর্কিত ধারণাসমূহ:

- **Agent Middleware** - এজেন্ট কলগুলো আটকান (পূর্ববর্তী নোটবুক)
- **Workflow State Management** - রানগুলোর মধ্যে ওয়ার্কফ্লো স্টেট সংরক্ষণ করুন
- **Multi-Agent Collaboration** - মানব-ইন-দ্য-লুপকে এজেন্ট টিমের সাথে মিলান
- **Event-Driven Architectures** - ইভেন্টের মাধ্যমে প্রতিক্রিয়াশীল সিস্টেম তৈরি করুন

---

### 🎓 অভিনন্দন!

আপনি Microsoft Agent Framework দিয়ে মানব-ইন-দ্য-লুপ ওয়ার্কফ্লো সম্পূর্ণরূপে আয়ত্ত করেছেন! এখন আপনি জানতে পারেন কিভাবে:
- ✅ মানবিক ইনপুট সংগ্রহের জন্য ওয়ার্কফ্লো থামাবেন
- ✅ RequestInfoExecutor এবং RequestInfoMessage ব্যবহার করবেন
- ✅ ইভেন্টের মাধ্যমে স্ট্রিমিং এক্সিকিউশন পরিচালনা করবেন
- ✅ @handler সহ কাস্টম এক্সিকিউটর তৈরি করবেন
- ✅ মানব সিদ্ধান্তের ভিত্তিতে ওয়ার্কফ্লো রাউট করবেন
- ✅ মানবদের সাথে সহযোগিতায় ইন্টারেক্টিভ AI এজেন্ট তৈরি করবেন

**এটি নির্ভরযোগ্য ও নিয়ন্ত্রণযোগ্য AI সিস্টেম তৈরির জন্য একটি গুরুত্বপূর্ণ প্যাটার্ন!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
